In [7]:
import os

import openai

from qdrant_client import QdrantClient
from langsmith import Client as LSClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Downlaod an example reference dta point from LangSmith

In [8]:
ls_client = LSClient(api_key=os.getenv("LANGSMITH_API_KEY"))

In [9]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [11]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('0d25e306-6d12-4aff-ae99-aa78b96b4bb4'), created_at=datetime.datetime(2026, 4, 12, 16, 58, 53, 380501, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 4, 12, 16, 58, 53, 380501, tzinfo=TzInfo(0)), example_count=33, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.3-arm64-arm-64bit-Mach-O', 'sdk_version': '0.7.30', 'runtime_version': '3.14.3', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [19]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[16].outputs

{'ground_truth': 'Shorn, Monsters of Midlife, Serving Absynthe, A Dark Age Resurgent, Devil of Dublin, and Ember Shadows and the Fate of Mount Neve all include fantasy, supernatural, or otherworldly elements.',
 'reference_context_ids': ['B09XGZV7YT',
  'B0B6LSBPR1',
  'B09WRYDH3T',
  'B0BB66596C',
  'B0BFW7MV1C',
  '1510109951'],
 'reference_descriptions': ['Shorn (The Sky Seekers) "AN INTELLIGENT AND CAPTIVATING NOVEL BY A NEW WRITER OF PROMISE." --Victoria Strauss, author of The Burning Land Jhared Denaban, a cursed soldier, enters his country\'s service desperate to pay the generations-old debt caused by his race’s betrayal. Though Shorn of his wings, like all his treacherous kind, Jhared still burns with dark desires for rebellion and flight. Posted to the border to protect Avelos from foreign invaders, he discovers that his best intentions inevitably turn toward chaos. As he seeks the source of grave political and magical threats, he must defend himself from the people who believ

In [20]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[16].inputs

{'question': 'Which books would appeal to readers who like dark fantasy or supernatural stories?'}

In [21]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[16].inputs
reference_outputs = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[16].outputs

### RAG Pipeline

In [24]:
def get_embedding(text, model='text-embedding-3-small'):

    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

def RAG_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocess_context = process_context(retrieved_context)
    prompt = build_prompt(preprocess_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [25]:
RAG_pipeline("Which books would appeal to readers who like dark fantasy or supernatural stories?")

{'answer': 'The available products that would likely appeal to readers who like dark fantasy or supernatural stories are:\n\n- A Dark Age Resurgent (WTF:ADAR / We The Fallen) – dark science-fantasy with horror and cosmic dread.\n- Serving Absynthe: Monstrous Hearts Absynthe Cleaver, Necromancer and Villainess – dark paranormal romance featuring monsters/supernatural themes.\n- Devil of Dublin: A Dark Irish Mafia Romance – very dark romantic suspense with Irish folklore and fairy-tale dark elements.\n- Monsters of Midlife: A Paranormal Women’s Fiction Trilogy (My Ex-Husband, the God of Monsters) – paranormal/dark fantasy romance centered on demigods and monsters.\n- Shorn (The Sky Seekers) – a dark fantasy with cursed characters, political/magical threats, and high-stakes supernatural intrigue.',
 'question': 'Which books would appeal to readers who like dark fantasy or supernatural stories?',
 'retrieved_context_ids': ['B0BB66596C',
  'B09WRYDH3T',
  'B0BFW7MV1C',
  'B0B6LSBPR1',
  'B0

### RAGAS metrics

In [28]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/nt/r1stvkv52mx2bbwlpbgbnlsm0000gp/T/ipykernel_39463/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/nt/r1stvkv52mx2bbwlpbgbnlsm0000gp/T/ipykernel_39463/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/nt/r1stvkv52mx2bbwlpbgbnlsm0000gp/T/ipykernel_39463/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [44]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/nt/r1stvkv52mx2bbwlpbgbnlsm0000gp/T/ipykernel_39463/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/nt/r1stvkv52mx2bbwlpbgbnlsm0000gp/T/ipykernel_39463/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [31]:
result = RAG_pipeline("Which books would appeal to readers who like dark fantasy or supernatural stories?")

In [37]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [38]:
await ragas_context_precision_id_based(result, reference_outputs)

1.0

In [39]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [40]:
await ragas_context_recall_id_based(result, reference_outputs)

0.8333333333333334

In [41]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [45]:
await ragas_faithfulness(result, reference_outputs)

0.8333333333333334

In [46]:
async def ragas_response_relevancy(run, example):
    
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [48]:
await ragas_response_relevancy(result, reference_outputs)

np.float64(0.7865178795589903)